# Example 7.9.1 / 1: Build an Explicit Diagnostic Model

Companion notebook for **7.9.1 Feasibility Relaxation** in the AMPL lab Documentation, section "Build an Explicit Diagnostic Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Build an Explicit Diagnostic Model

In the steel model, firm orders `commit[p]` need 43.66 hours, but only 40 are available, so the
model with `Make[p] >= commit[p]` is infeasible. Add a nonnegative violation variable to each rule
that may give way, and minimize the weighted violations: here an hour of overtime is penalized at
1,000 and a ton of unmet order at 1:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param market {PROD} >= 0;         # most that can be sold, in tons
param commit {PROD} >= 0;         # tons already ordered
param avail >= 0;                 # hours available
param overtime_penalty > 0;       # penalty per hour beyond avail
param short_penalty > 0;          # penalty per ton of unmet order

var Make {p in PROD} >= 0, <= market[p];  # tons produced
var Overtime >= 0;                         # violation of the hour limit, hours
var Short {PROD} >= 0;                     # violation of each order, tons

# weighted violation; zero means the original rules hold
minimize TotalViolation:
    overtime_penalty * Overtime + sum {p in PROD} short_penalty * Short[p];
# original rules, each loosened by its violation
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail + Overtime;
subject to Orders {p in PROD}: Make[p] >= commit[p] - Short[p];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["commit"] = {"bands": 4000, "coils": 2000, "plate": 1500}
ampl.param["avail"] = 40
ampl.param["overtime_penalty"] = 1000
ampl.param["short_penalty"] = 1

In [5]:
# run.py
ampl.solve(solver="highs")
print("Overtime:", ampl.var["Overtime"].value())
print("Short:", ampl.var["Short"].to_dict())

HiGHS 1.14.0: optimal solution; objective 512.5
4 simplex iterations
0 barrier iterations
Overtime: 0.0
Short: {'bands': 0, 'coils': 512.4999999999999, 'plate': 0}


### Pure AMPL: data.dat and model.run

With these penalties the cheapest repair is to leave about 512.5 tons of the coils order unmet:
an hour makes only 140 tons of coils, against 160 of plate and 200 of bands, so shorting coils
frees the missing 3.66 hours at the lowest cost in tons. Choose penalties in units that reflect
business importance; a weighted sum trades violations against one another and does not
necessarily minimize the number of violated rules.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: PROD: rate market commit :=
  bands  200  6000  4000
  coils  140  4000  2000
  plate  160  3500  1500 ;
param avail := 40;
param overtime_penalty := 1000;
param short_penalty := 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Overtime, Short;

HiGHS 1.14.0: optimal solution; objective 512.5
4 simplex iterations
0 barrier iterations
Overtime = 0

Short [*] :=
bands    0
coils  512.5
plate    0
;

